# 08 · Inference Demonstration

**Artemis-X - AI perception for lunar rock and obstacle detection**

**Objective.** Load the frozen detector and run it with exactly the frozen settings, on GPU when available and otherwise on CPU. Only a small demonstration inference is performed: no training and no dataset processing.

**Research questions**

1. How is the frozen model loaded and verified?
2. What does its output look like?
3. How fast is it?

**Inputs (published, pixel-free artifacts)**

- `models/artemis_rock_yolo11n.pt`
- `artifacts/provenance/FROZEN_MODEL_CONFIG.json`

> **Provisional results.** All labels were produced by an automated pipeline (S2-H policy) and have **not** been verified by humans. Metrics are measured on synthetic renders only and do not establish performance on real lunar imagery or readiness for rover deployment.

## Setup
The notebook imports the `artemis_x` package from `src/` and reads only published artifacts. It never trains a model, rebuilds a dataset or evaluates the held-out test split.

In [1]:
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "artemis_x").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from artemis_x.config import paths
from artemis_x.reporting import artifacts as A, plots as P
P.use_style()
pd.set_option("display.precision", 3); pd.set_option("display.max_colwidth", 90)
# Dataset pixels are never committed (redistribution licence unconfirmed). Set ARTEMIS_SHOW_LOCAL_IMAGES=1 to view
# local images when the dataset folder is present.
SHOW_LOCAL_IMAGES = os.environ.get("ARTEMIS_SHOW_LOCAL_IMAGES") == "1" and paths.local_dataset_available()
FIG = paths.ARTIFACTS / "figures"
print("artemis_x ready | local dataset present:", paths.local_dataset_available(), "| showing local images:", SHOW_LOCAL_IMAGES)

artemis_x ready | local dataset present: True | showing local images: False


## 1. Load and verify the frozen model
`load_frozen_spec()` reads `FROZEN_MODEL_CONFIG.json` and refuses to continue if the weights' SHA-256 differs from the frozen value.

In [2]:
from artemis_x.ml.model_registry import load_frozen_spec
from artemis_x.ml.inference import RockDetector, default_device
spec = load_frozen_spec()
pd.Series({"weights": paths.rel(spec.weights), "sha256 (verified)": spec.sha256, "class names": spec.class_names,
           "imgsz": spec.imgsz, "confidence": spec.conf, "NMS IoU": spec.nms_iou, "max detections": spec.max_det,
           "min box area (px^2)": spec.min_box_area_px2, "selected epoch": spec.epoch, "device": default_device()}).to_frame("frozen detector")

,frozen detector
weights,models/artemis_rock_yolo11n.pt
sha256 (verified),413d0481a728a310f5b919d069676ee5ddd2091e81b4b663831e1e977a0752eb
class names,{0: 'rock'}
imgsz,736
confidence,0.2
NMS IoU,0.6
max detections,300
min box area (px^2),50.0
selected epoch,111
device,cuda:0


In [3]:
det = RockDetector(spec)          # CUDA if available, else CPU
det.warmup()

## 2. Example inputs
**No redistribution-cleared lunar images are bundled with this repository.** The licence of the Artificial Lunar Landscape renders
has not been confirmed, so no dataset pixels are committed. To try the model on real inputs:
1. download the Artificial Lunar Landscape dataset (see `docs/DATASET.md`) into `dataset/artificial-lunar/`, or
2. point `IMAGE_PATHS` below at your own 720 x 480 lunar images.

Run with `ARTEMIS_SHOW_LOCAL_IMAGES=1` to draw the detections. Without images, the cell below runs a **synthetic** frame only, to
demonstrate the API and timing. A blank frame is expected to produce no detections.

In [4]:
import numpy as np, time
IMAGE_PATHS = []
if paths.local_dataset_available():
    # validation scenes only - the held-out test split is never used for demonstrations
    IMAGE_PATHS = [paths.YOLO_DATASET / "images" / "val" / f"render{i:04d}.png" for i in (115, 5589, 575)]
rows = []
for p in IMAGE_PATHS:
    res = det.predict(p)
    c = res.as_array()[:, 4]
    rows.append({"image": p.name, "detections": len(res.detections), "removed < 50 px^2": res.removed_small_boxes,
                 "max conf": round(float(c.max()), 3) if len(c) else None, "median conf": round(float(np.median(c)), 3) if len(c) else None,
                 "latency (ms)": round(res.inference_ms, 1), "device": res.device})
    if SHOW_LOCAL_IMAGES:
        from PIL import Image
        P.draw_detections(np.asarray(Image.open(p).convert("RGB")), res.as_array(), f"{p.name}: {len(res.detections)} rocks @ conf >= {spec.conf}")
synthetic = det.predict(np.full((480, 720, 3), 90, dtype=np.uint8))
rows.append({"image": "synthetic grey frame", "detections": len(synthetic.detections), "removed < 50 px^2": synthetic.removed_small_boxes,
             "max conf": None, "median conf": None, "latency (ms)": round(synthetic.inference_ms, 1), "device": synthetic.device})
pd.DataFrame(rows)

,image,detections,removed < 50 px^2,max conf,median conf,latency (ms),device
0,render0115.png,10,0,0.844,0.351,23.4,cuda:0
1,render5589.png,147,0,0.899,0.377,15.6,cuda:0
2,render0575.png,45,0,0.830,0.416,14.0,cuda:0
3,synthetic grey frame,0,0,NaN,NaN,8.9,cuda:0


## 3. Latency
Median of 20 single-image runs. This includes Ultralytics pre- and post-processing, but not disk I/O.

In [5]:
frame = np.random.default_rng(0).integers(0, 255, (480, 720, 3), dtype=np.uint8)
def bench(d, n=20):
    ts = []
    for _ in range(n):
        t = time.perf_counter(); d.predict(frame); ts.append((time.perf_counter() - t) * 1000)
    return float(np.median(ts)), float(np.percentile(ts, 95))
out = {det.device: bench(det)}
if det.device != "cpu":
    out["cpu"] = bench(RockDetector(spec, device="cpu"), n=5)
pd.DataFrame(out, index=["median ms", "p95 ms"]).T.round(1)

,median ms,p95 ms
cuda:0,8.6,10.1
cpu,20.9,64.7


Historical reference: on the RTX 5060 Laptop GPU, the Task F parity check measured a median of 15.3 ms per 720 x 480 image,
including file I/O. Live inference reproduced the cached validation predictions exactly (1,904 of 1,904 boxes on 50 images).

**Using the detector in code**
```python
from artemis_x.ml.inference import RockDetector
det = RockDetector()                    # verifies the frozen SHA-256, picks CUDA or CPU
result = det.predict("frame.png")       # frozen imgsz/conf/NMS/max_det + 50 px^2 post-filter
for d in result.detections: print(d.x0, d.y0, d.x1, d.y1, d.confidence)
```

## Provenance
Each artifact below is a verified copy (or path-sanitised derivative) of a historical experiment output; `scripts/verify_artifacts.py` re-checks every hash.

In [6]:
A.provenance(['provenance/FROZEN_MODEL_CONFIG.json', 'evaluation/inference_check.json'])

,artifact,mode,source (local experiments/),sha256
0,artifacts/provenance/FROZEN_MODEL_CONFIG.json,COPY,experiments/task-f/model-selection-001/FROZEN_MODEL_CONFIG.json,efdc5802cbc3a4f0...
1,artifacts/evaluation/inference_check.json,COPY,experiments/task-f/model-selection-001/inference_check.json,fc1ed987d65d5173...


## Conclusion
- The frozen detector loads with checksum verification and runs on CPU or GPU through one call.
- Future Webots integration (planned, not implemented) will feed simulator camera frames to the same `RockDetector`. A domain-gap
  check on Webots frames must come first; see `future/WEBOTS_INTEGRATION_PLAN.md`.